In [ ]:
import pandas as pd
import os

from adodbapi.is64bit import Python

In [ ]:
base_dir = "../data/raw/Imports-Exports"

file_map = {
    "df_can": "CAN_EX_IM.csv",
    "df_chi": "CHI_EX_IM.csv",
    "df_ger": "GER_EX_IM.xlsx",
    "df_jap": "JAPAN_EX_IM.xlsx",
    "df_usa": "US_EX_IM.xlsx",
    "df_mex_ex": "MEX_EX.csv",
    "df_mex_im": "MEX_IM.csv"
}

In [ ]:
# Load with correct separator and decimal/thousands handling
df = pd.read_csv("../data/raw/Imports-Exports/CAN_EX_IM.csv", index_col=0, thousands=",")  # or .xlsx if applicable

# Transpose so that months become rows
df = df.T.reset_index().rename(columns={"index": "date"})

# Optional: Convert date strings like "Jan-97" to datetime (first of month)
df["date"] = pd.to_datetime(df["date"], format="%b-%y")
df = df.rename(columns={"Export": "EX_M_CAN", "Import": "IM_M_CAN"})
df.drop(columns=["Trade Balance"], inplace=True)
# Done! Now you have:
# - columns: date, Import, Export, Trade Balance
# - one row per month

display(df.head())
df.to_csv("../data/processed/CLEAN_CAN_EX_IM.csv")

In [ ]:
# Load the file
df_ger = pd.read_excel("../data/raw/Imports-Exports/GER_EX_IM.xlsx")
print(df_ger.columns)
# Combine Year and Month into a datetime
df_ger["date"] = pd.to_datetime(df_ger["Year"].astype(str) + " " + df_ger["Month"], format="%Y %B")

# Keep only relevant columns
df_ger = df_ger[["date", "Exports", "Imports"]]

# Optional: Convert to millions if needed
df_ger["Exports"] = df_ger["Exports"] / 1_000_000
df_ger["Imports"] = df_ger["Imports"] / 1_000_000

# Rename columns to reflect country
df_ger = df_ger.rename(columns={"Exports": "EX_M_GER", "Imports": "IM_M_GER"})

# Sort by date
df_ger = df_ger.sort_values("date").reset_index(drop=True)
df_ger.to_csv("../data/processed/CLEAN_GER_EX_IM.csv")
display(df_ger.head())

In [ ]:
# Load the XLSX file (it's a single-column CSV inside Excel)
df_JAP_raw = pd.read_excel("../data/raw/Imports-Exports/JAP_EX_IM.xlsx", header=None)

# Split the single string column by commas into 3 separate columns
data = df_JAP_raw[0].str.split(",", expand=True)
data.columns = ["date", "Exports", "Imports"]

# Drop the header row ("Years/Months", etc.)
data = data[1:].copy()

# Convert to proper formats
data["date"] = pd.to_datetime(data["date"], format="%Y/%m")
data["Exports"] = pd.to_numeric(data["Exports"].str.strip(), errors="coerce") / 1_000_000
data["Imports"] = pd.to_numeric(data["Imports"].str.strip(), errors="coerce") / 1_000_000

# Rename columns
df_JAP = data.rename(columns={"Exports": "EX_M_JAP", "Imports": "IM_M_JAP"})

# Optional: sort by date
df_JAP = df_JAP.sort_values("date").reset_index(drop=True)
df_JAP.to_csv("../data/processed/CLEAN_JAP_EX_IM.csv")
display(df_JAP.head())

In [ ]:
# Load the data
df_mex_ex = pd.read_csv("../data/raw/Imports-Exports/MEX_EX.csv")
print(df_mex_ex.columns)
# Parse date column properly if not already
df_mex_ex["date"] = pd.to_datetime(df_mex_ex["date"])

# Rename for consistency
df_mex_ex = df_mex_ex.rename(columns={"value": "EX_M_MEX"})

# Optional: sort by date
df_mex_ex = df_mex_ex.sort_values("date").reset_index(drop=True)

display(df_mex_ex.head())

In [ ]:
# Load the data
df_mex_im = pd.read_csv("../data/raw/Imports-Exports/MEX_IM.csv")
print(df_mex_im.columns)
# Parse date column properly if not already
df_mex_im["date"] = pd.to_datetime(df_mex_im["date"])

# Rename for consistency
df_mex_im = df_mex_im.rename(columns={"value": "IM_M_MEX"})

# Optional: sort by date
df_mex_im = df_mex_im.sort_values("date").reset_index(drop=True)

display(df_mex_im.head())

In [ ]:
# Merge on 'date'
df_mex = pd.merge(df_mex_ex, df_mex_im, on="date", how="outer").sort_values("date").reset_index(drop=True)

df_mex.to_csv("../data/processed/CLEAN_MEX_EX_IM.csv")
display(df_mex.head())

In [ ]:
# Load the file
df_usa = pd.read_excel("../data/raw/Imports-Exports/USA_SHORT_EX_IM.xlsx")

# Rename columns for simplicity
df_usa = df_usa.rename(columns={
    "exports (millions USD)": "EX_M_USA",
    "imports (millions USD)": "IM_M_USA"
})

# Clean the 'date' column first
df_usa["date"] = df_usa["date"].astype(str).str.strip()

# Parse cleaned dates
df_usa["date"] = pd.to_datetime(df_usa["date"], format="%Y %b", errors="coerce")

# Remove commas and convert to numeric
df_usa["EX_M_USA"] = pd.to_numeric(df_usa["EX_M_USA"].astype(str).str.replace(",", ""), errors="coerce")
df_usa["IM_M_USA"] = pd.to_numeric(df_usa["IM_M_USA"].astype(str).str.replace(",", ""), errors="coerce")

# Optional: drop invalid rows and sort
df_usa = df_usa.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)
df_usa.to_csv("../data/processed/CLEAN_USA_EX_IM.csv")
display(df_usa.head())

In [ ]:
# Step 1: Load original column headers as date strings
original_columns = pd.read_csv("../data/raw/Imports-Exports/CHI_EX_IM.csv", nrows=0, encoding="latin1", engine="python", sep=None).columns.tolist()

# Step 2: Re-run your transpose logic
df_chi = pd.read_csv("../data/raw/Imports-Exports/CHI_EX_IM.csv", header=None, encoding="latin1", engine="python", sep=None)
df_chi = df_chi.T
df_chi.columns = df_chi.iloc[0]
df_chi = df_chi.drop(index=0).reset_index(drop=True)

# Step 3: Rename and assign parsed dates
df_chi = df_chi.rename(columns={
    "Total Value of Exports Current Period(1000 US dollars)": "EX_M_CHI",
    "Total Value of Imports Current Period(1000 US dollars)": "IM_M_CHI"
})
df_chi["date"] = pd.to_datetime(original_columns[1:], format="%b-%y", errors="coerce")  # skip 'Indicators' column

# Step 4: Keep only relevant columns and convert
df_chi = df_chi[["date", "EX_M_CHI", "IM_M_CHI"]]
df_chi["EX_M_CHI"] = pd.to_numeric(df_chi["EX_M_CHI"], errors="coerce") / 1_000
df_chi["IM_M_CHI"] = pd.to_numeric(df_chi["IM_M_CHI"], errors="coerce") / 1_000
df_chi = df_chi.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)

df_chi.to_csv("../data/processed/CLEAN_CHI_EX_IM.csv")
display(df_chi.head())

In [ ]:
import pandas as pd

# Load the file
df = pd.read_excel("../data/raw/Mexico_Policy_Uncertainty_Data.xlsx")

# Create 'date' column as the first day of each month
df["date"] = pd.to_datetime(df[["Year", "Month"]].assign(DAY=1))

# Rename the EPU column
df = df.rename(columns={"EPU_MEX": "EPU_M_MEX"})

# Keep only the relevant columns
df = df[["date", "EPU_M_MEX"]]

# Optional: sort by date just in case
df = df.sort_values("date").reset_index(drop=True)

# Show or save
print(df.head())
df.to_csv("../data/processed/EPU_MEX_cleaned.csv", index=False)


In [ ]:
df = pd.read_excel("../data/manual-data/GER_EX_IM_USD.xlsx")

In [ ]:
df

In [ ]:
df = pd.read_excel("../data/manual-data/GER_EX_IM_USD.xlsx")

# Clean 'Year' column (remove junk, convert to numeric)
df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
df["Year"] = df["Year"].ffill().astype(int)

# Normalize 'Month' column: convert to string, then get month number
df["Month"] = df["Month"].astype(str).str.strip()

# Try converting both names and numbers to month numbers
def parse_month(m):
    try:
        return pd.to_datetime(m, format="%B").month  # e.g. January
    except:
        try:
            return int(m)  # e.g. 1 or "01"
        except:
            return pd.NA

df["Month"] = df["Month"].apply(parse_month)

# Drop rows where month couldn't be parsed
df = df.dropna(subset=["Month"])
df["Month"] = df["Month"].astype(int)

# Create the date column
df["date"] = pd.to_datetime(df[["Year", "Month"]].assign(DAY=1)).dt.to_period("M").dt.to_timestamp()

# Final DataFrame
result_df = df[["date", "EX_GER", "EX_GER"]]
df.to_csv("../data/manual-data/CLEAN_EX_IM.csv", index=False)
print(result_df.head())


In [ ]:
# Load the data
df = pd.read_csv("../data/raw/manual-data/CLEAN_EX_IM.csv")

# Convert to billions
df["EX_GER"] = df["EX_GER"] / 1_000_000
df["IM_GER"] = df["IM_GER"] / 1_000_000

# Save if needed
df.to_csv("../data/processed/CLEAN_EX_IM_B.csv", index=False)

df

In [ ]:
df_ex = pd.read_excel("../data/manual-data/EX_GER_USD.xlsx")
df_im = pd.read_excel("../data/manual-data/IM_GER_USD.xlsx")

print(df_ex)

In [ ]:
df_ex["EX_GER"] = (df_ex["EX_GER"] / 1_000_000_000).round(4)
df_im["IM_GER"] = (df_im["IM_GER"] / 1_000_000_000).round(4)

df_ex

In [ ]:
df_ex.to_csv("../data/processed/EX_GER_USD.csv", index=False)
df_im.to_csv("../data/processed/IM_GER_USD.csv", index=False)